# 写一个简单的聊天机器
- 实现功能：
> 与大语言模型多轮对话
>
> 流式输出（假流式）
>
> 工具调用
>

In [5]:
import os
from openai import OpenAI
import json

# 在根目录下创建.env 文件写入API_KEY 和 BASE_URL
# 这样可以不用配置电脑环境变量（注意不要把.env提交到git上了）
# 这行会把 .env 文件里的内容加载到 os.environ 中
from dotenv import load_dotenv

load_dotenv() # 把 .env 文件里的内容加载到 os.environ 中

# 从环境变量读取，避免把 API Key 硬编码到代码里
API_KEY = os.environ.get("API_KEY")
BASE_URL = os.environ.get("BASE_URL")
if not API_KEY or not BASE_URL:
    raise RuntimeError("请先设置环境变量 API_KEY 和 BASE_URL")

### 获取用户输入
> 1. 创建一个简易的用户消息类（只包含prompt，思考模式，和思考强度）
>
> 2. 创建一个消息获取函数，每一轮对话调用该函数即可
>

In [6]:
class UserMessage:
    def __init__(self, user_msg, thinking_mode="disabled", thinking_strength = "medium"):
        self.user_msg = user_msg
        self.thinking_mode = thinking_mode
        self.thinking_strength = thinking_strength

In [7]:
# 接受用户信息
def build_user_message():

    # 1. 用户的 prompt
    user_input=input("你：")
    if user_input == "q":
        return "q"

    message_history.append({"role": "user", "content": user_input})

    # 思考模式相关（是否开启思考，思考强度）
    # DS的思考模式不支持 temperature、presence_penalty、frequency_penalty 参数
    thinking_option="disabled"
    user_thinking_option = input("是否启用思考模式？(y/n): ").strip().lower()
    if user_thinking_option == "y":
        thinking_option = "enabled"

    # 2. 思考强度选择
    thinking_strength = "medium"  # 默认中等强度
    if thinking_option == "enabled":
        user_strength_option = input("选择思考强度 (low/medium/high): ").strip().lower()
        if user_strength_option in ["low", "medium", "high"]:
            thinking_strength = user_strength_option
        else:
            print("无效的强度选项，使用默认值 medium。")

    return UserMessage(user_input, thinking_option, thinking_strength)

### 工具设置
> 1. 用`json`格式描述可以使用的工具
>
> 2. 调用`tools` 包里面的工具
>
> 3. 设置工具名到函数的映射
>

In [10]:
# 用 JSON 格式描述模型可以使用的工具
from tools import regular_tools

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的当前天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# 工具名到函数的dict映射
tool_functions = {"get_weather": regular_tools.get_weather}

### 向模型发起请求并获得输入
> 1. 两种方式，一种流式输出，一种非流式输出
>
> 2. 在得到模型输出的同时，把结果写入会话历史
>
> 3. 同时打印结果
>
> 4. 新增加一种调用工具的回复方式
>

In [11]:
def get_response(user_message, client):
    global message_history

    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=message_history,
        reasoning_effort=user_message.thinking_strength,
        extra_body={"thinking": {"type": user_message.thinking_mode}}
    )

    reply = response.choices[0].message.content
    # reasoning_content = response.choices[0].message.reasoning_content
    reasoning_content = getattr(response.choices[0].message, "reasoning_content", None)

    # 官方文档的写法：messages.append(response.choices[0].message)
    message_history.append({"role": "assistant", "content": reply})

    print(f"AI: {reply}\n")

In [12]:
def get_response_stream(user_message, client):

    global message_history

    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=message_history,
        reasoning_effort=user_message.thinking_strength,
        stream=True,
        extra_body={"thinking": {"type": user_message.thinking_mode}}
    )

    reply = ""
    reasoning_content = ""

    print("AI:",end="")
    for chunk in response:
        delta = chunk.choices[0].delta
        delta_reasoning = getattr(delta, "reasoning_content", None)
        if delta_reasoning:
            reasoning_content += delta_reasoning
        else:
            reply_part = delta.content
            if reply_part is not None:
                reply += reply_part
                print(f"{reply_part}", end="", flush=True)

    # 模型默认会忽略掉 reasoning_content
    message_history.append({"role": "assistant", "reasoning_content": reasoning_content, "content": reply})

- 工具调用相关问题
> 1. 检查是否调用工具
>
> 2. 解析大模型返回的 `arguments` JSON串
>
> 3. 执行该工具对应的函数
> 4. 将执行结果重新发给大模型再生成一次对话

In [13]:
def get_response_with_tools(user_message, client):
    global tools
    global message_history

    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=message_history,
        tools=tools,
        reasoning_effort=user_message.thinking_strength,
        extra_body={"thinking": {"type": user_message.thinking_mode}}
    )

    assistant_message = response.choices[0].message

    # 如果模型决定调用工具
    if assistant_message.tool_calls:
        message_history.append(assistant_message)
        for tool_call in assistant_message.tool_calls:
            # arguments 是 JSON 字符串，需要解析成字典
            args = json.loads(tool_call.function.arguments)
            # **args 把字典解包成关键字参数，等价于 func(city="北京")
            func = tool_functions[tool_call.function.name]
            result = func(**args)
            print(f"  [调用工具] {tool_call.function.name}({args}) => {result}")
            # role 为 "tool" 表示这是工具返回的结果
            # tool_call_id 用来关联这条结果对应哪个工具调用
            message_history.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        # 拿到工具结果后再调一次模型，让它生成自然语言回答
        response = client.chat.completions.create(
            model="deepseek-flash",
            messages=message_history,
            reasoning_effort=user_message.thinking_strength,
            extra_body={"thinking": {"type": user_message.thinking_mode}}
        )
        assistant_message = response.choices[0].message

    message_history.append({"role": "assistant", "content": assistant_message.content})
    print(f"AI: {assistant_message.content}\n")

### 基础设置
> 1. 创建 `client` 对象
> 2. 初始化聊天记录
> 3. 输入系统提示词

In [14]:
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
SYSTEM_PROMPT = "You`re a funny guy who always answer in Chinese"

message_history = [{
    "role": "system",
    "content": SYSTEM_PROMPT
}]

print(f"[人设]{SYSTEM_PROMPT}")
print("输入消息开始聊天，输入 q 退出\n")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
SYSTEM_PROMPT = "You`re a funny guy who always answer in Chinese"

# 本轮对话历史
message_history = [{
    "role": "system",
    "content": SYSTEM_PROMPT
}]

[人设]You`re a funny guy who always answer in Chinese
输入消息开始聊天，输入 q 退出



### 开始执行

In [15]:
while True:
    user_message = build_user_message()
    if user_message == "q":
        break

    is_stream = input("选择流式输出(y/n): ").strip().lower()
    if is_stream == "y":
        get_response_stream(user_message, client)
        print()
    elif is_stream == "n":
        get_response_with_tools(user_message, client)
    else:
        print("输入无效，默认采用流式输出")
        get_response_stream(user_message, client)

  [调用工具] get_weather({'city': '北京'}) => {"temperature": 8, "condition": "多云"}
AI: 北京现在8度，多云。出门记得穿暖和点，别冻着自己！

